In [2]:
def matriz_espiral(n):
    matriz = [[0] * n for _ in range(n)]

    arriba, abajo = 0, n - 1
    izquierda, derecha = 0, n - 1
    num = 1

    while arriba <= abajo and izquierda <= derecha:
        # Izquierda -> derecha (fila superior)
        for col in range(izquierda, derecha + 1):
            matriz[arriba][col] = num
            num += 1
        arriba += 1

        # Arriba -> abajo (columna derecha)
        for fila in range(arriba, abajo + 1):
            matriz[fila][derecha] = num
            num += 1
        derecha -= 1

        # Derecha -> izquierda (fila inferior)
        if arriba <= abajo:
            for col in range(derecha, izquierda - 1, -1):
                matriz[abajo][col] = num
                num += 1
            abajo -= 1

        # Abajo -> arriba (columna izquierda)
        if izquierda <= derecha:
            for fila in range(abajo, arriba - 1, -1):
                matriz[fila][izquierda] = num
                num += 1
            izquierda += 1

    return matriz


def imprimir_matriz(matriz):
    ancho = len(str(len(matriz) ** 2)) + 1
    for fila in matriz:
        print("".join(f"{x:>{ancho}}" for x in fila))


for tam in (4, 9):
    print(f"Matriz de tamaño {tam}:")
    imprimir_matriz(matriz_espiral(tam))
    print()

Matriz de tamaño 4:
  1  2  3  4
 12 13 14  5
 11 16 15  6
 10  9  8  7

Matriz de tamaño 9:
  1  2  3  4  5  6  7  8  9
 32 33 34 35 36 37 38 39 10
 31 56 57 58 59 60 61 40 11
 30 55 72 73 74 75 62 41 12
 29 54 71 80 81 76 63 42 13
 28 53 70 79 78 77 64 43 14
 27 52 69 68 67 66 65 44 15
 26 51 50 49 48 47 46 45 16
 25 24 23 22 21 20 19 18 17



In [3]:
import re
from fractions import Fraction

# Término: número o x, con signo opcional (ej: 3, +4, -x, x)
TERMINO = r"[+-]?(?:\d+|x)"
# Expresión: primer término seguido de grupos [+-]término (sin signos dobles)
EXPRESION = TERMINO + r"(?:[+-](?:\d+|x))*"

# El lookbehind/lookahead evita coger trozos de algo mayor
# (3x+5=11, ++x-3=x+6, x=--x...)
PATRON_ECUACION = re.compile(
    rf"(?<![\w+\-=])({EXPRESION})=({EXPRESION})(?![\w+\-=])"
)

PATRON_TERMINO = re.compile(r"([+-]?)(\d+|x)")


def buscar_ecuaciones(texto):
    """Devuelve todas las ecuaciones válidas encontradas en el texto."""
    return [m.group() for m in PATRON_ECUACION.finditer(texto)]


def analizar_lado(lado):
    """Devuelve (coeficiente de x, término independiente) de un lado."""
    coeficiente_x = 0
    constante = 0
    for m in PATRON_TERMINO.finditer(lado):
        signo = -1 if m.group(1) == "-" else 1
        if m.group(2) == "x":
            coeficiente_x += signo
        else:
            constante += signo * int(m.group(2))
    return coeficiente_x, constante


def resolver(ecuacion):
    """Resuelve una ecuación de la forma ladoIzq=ladoDer."""
    izquierdo, derecho = ecuacion.split("=")
    a1, b1 = analizar_lado(izquierdo)
    a2, b2 = analizar_lado(derecho)

    coeficiente = a1 - a2
    independiente = b2 - b1

    if coeficiente == 0:
        return "infinitas soluciones" if independiente == 0 else "sin solución"

    solucion = Fraction(independiente, coeficiente)
    return str(solucion.numerator) if solucion.denominator == 1 else str(float(solucion))


def main():
    texto = """\
Ecuaciones de primer grado con una incógnita. Veamos algunos ejemplos válidos y no válidos:
Ejercicio1: x=3+13
Ejercicio2: 8+70=x
Ejercicio3: -x+2=x-8
Ejercicio4: x-1+x+3=2+x+8-x
Ejercicio5: +4+x-3-x+1+x=x+x+x-1-1
No vale: ++x-3=x+6
No vale: x++5=8+x
No vale: x=--x
No vale: 3-1+x=-4+6--x
No vale: 3x+5=11
No vale: x-3=2x-8
"""
    for ecuacion in buscar_ecuaciones(texto):
        print(f"{ecuacion:<24} Solución: {resolver(ecuacion)}")


if __name__ == "__main__":
    main()

x=3+13                   Solución: 16
8+70=x                   Solución: 78
-x+2=x-8                 Solución: 5
x-1+x+3=2+x+8-x          Solución: 4
+4+x-3-x+1+x=x+x+x-1-1   Solución: 2
